# Task 2 — Loan Approval Prediction

Place the official loan dataset in `data/raw/` with `loan` in its filename. Confirm the target column from the assignment before training.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
DATA = ROOT / "data" / "raw"
files = list(DATA.glob("*.csv")) + list(DATA.glob("*.xlsx")) + list(DATA.glob("*.json"))
print("Data files found:", [f.name for f in files])
print("Task 2 dataset: Loan approval data. Results are educational and are not a real-world lending decision system.")

def load_data(keywords):
    matches = [f for f in files if any(k in f.name.lower() for k in keywords)]
    if not matches:
        print("DATASET MISSING for:", keywords)
        return None
    f = matches[0]
    print("Loading:", f.name)
    if f.suffix.lower() == ".csv":
        return pd.read_csv(f)
    if f.suffix.lower() == ".xlsx":
        return pd.read_excel(f)
    return pd.read_json(f)

def overview(df):
    if df is None:
        return
    print("Shape:", df.shape)
    display(df.head())
    display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("missing"))
    display(df.describe(include="all").T)

df = load_data(['loan'])
overview(df)

Data files found: ['zomato.csv', 'loan_approval_dataset.csv', 'instagram_analytics.csv', 'business_sales_customers.csv']
Task 2 dataset: Loan approval data. Results are educational and are not a real-world lending decision system.
Loading: loan_approval_dataset.csv
Shape: (4269, 13)


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


,missing
loan_id,0
no_of_dependents,0
education,0
self_employed,0
income_annum,0
loan_amount,0
loan_term,0
cibil_score,0
residential_assets_value,0
commercial_assets_value,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
loan_id,4269.0,NaN,NaN,NaN,2135.0,1232.498479,1.0,1068.0,2135.0,3202.0,4269.0
no_of_dependents,4269.0,NaN,NaN,NaN,2.498712,1.69591,0.0,1.0,3.0,4.0,5.0
education,4269,2,Graduate,2144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
self_employed,4269,2,Yes,2150,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income_annum,4269.0,NaN,NaN,NaN,5059123.916608,2806839.831818,200000.0,2700000.0,5100000.0,7500000.0,9900000.0
loan_amount,4269.0,NaN,NaN,NaN,15133450.456781,9043362.984843,300000.0,7700000.0,14500000.0,21500000.0,39500000.0
loan_term,4269.0,NaN,NaN,NaN,10.900445,5.709187,2.0,6.0,10.0,16.0,20.0
cibil_score,4269.0,NaN,NaN,NaN,599.936051,172.430401,300.0,453.0,600.0,748.0,900.0
residential_assets_value,4269.0,NaN,NaN,NaN,7472616.537831,6503636.587664,-100000.0,2200000.0,5600000.0,11300000.0,29100000.0
commercial_assets_value,4269.0,NaN,NaN,NaN,4973155.305692,4388966.089638,0.0,1300000.0,3700000.0,7600000.0,19400000.0


## Baseline classification model

In [2]:
if df is not None:
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

    candidates = [c for c in df.columns if any(
        x in c.lower() for x in ["loan_status", "approved", "approval", "target"]
    )]
    print("Possible target columns:", candidates)
    if not candidates:
        print("TARGET NOT IDENTIFIED. Set target_col to the exact target column from the assignment.")
    else:
        target_col = candidates[0]
        work = df.dropna(subset=[target_col]).copy()
        X, y = work.drop(columns=[target_col]), work[target_col]
        if y.nunique() < 2:
            print("Target must contain at least two classes.")
        else:
            numeric_cols = X.select_dtypes(include="number").columns.tolist()
            cat_cols = X.select_dtypes(exclude="number").columns.tolist()
            pre = ColumnTransformer([
                ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                                  ("scale", StandardScaler())]), numeric_cols),
                ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                                  ("encode", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
            ])
            model = Pipeline([("preprocess", pre),
                              ("classifier", RandomForestClassifier(
                                  n_estimators=200, random_state=42,
                                  class_weight="balanced"))])
            strat = y if y.value_counts().min() >= 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=strat)
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            print("Target:", target_col)
            print(classification_report(y_test, pred, zero_division=0))
            print("Confusion matrix:\n", confusion_matrix(y_test, pred))
            print("Suggested threshold must be selected using validation data, costs of false approvals/rejections, and calibration—not guessed.")
else:
    print("Dataset missing. No model metrics or threshold can be truthfully reported yet.")

Possible target columns: [' loan_status']


Target:  loan_status
              precision    recall  f1-score   support

    Approved       0.98      0.98      0.98       531
    Rejected       0.98      0.97      0.97       323

    accuracy                           0.98       854
   macro avg       0.98      0.98      0.98       854
weighted avg       0.98      0.98      0.98       854

Confusion matrix:
 [[523   8]
 [ 10 313]]
Suggested threshold must be selected using validation data, costs of false approvals/rejections, and calibration—not guessed.


## Threshold selection and limitations

Select the operating threshold using a validation set, business costs, fairness checks and calibration. Do not use the test set repeatedly to tune it. This model is a learning exercise, not a real lending decision system.